# ForgeSight — Fine-tune Qwen2.5-3B on a free Colab T4

This notebook runs the QLoRA fine-tune end-to-end on a **free Colab T4 (16 GB)** and downloads the
GGUF you then serve locally via Ollama.

**Before you start:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU`, then
`Runtime → Run all`.

Pipeline: clone repo → install Unsloth → train (`finetune/colab_train.py`) → export `Q4_K_M.gguf`
→ download. Then locally: `ollama create qwen-forgesight -f finetune/Modelfile`.

In [1]:
#@title 1. Confirm the T4 GPU is attached
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


In [2]:
#@title 2. Clone the ForgeSight repo (gets the script + committed SFT dataset)
import os
if not os.path.isdir("ForgeSight"):
    !git clone --depth 1 https://github.com/Gurjas2112/ForgeSight.git
%cd ForgeSight
!wc -l finetune/dataset/sft_train.jsonl finetune/dataset/sft_eval.jsonl

Cloning into 'ForgeSight'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'ForgeSight'
/content
wc: finetune/dataset/sft_train.jsonl: No such file or directory
wc: finetune/dataset/sft_eval.jsonl: No such file or directory
0 total


In [ ]:
#@title 3. Install Unsloth + training stack (~2-3 min)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 22.9 MB/s eta 0:00:00


In [ ]:
#@title 4. Train + export GGUF (T4 defaults: max_seq=2048, bsz 2x4, 3 epochs)
# Building llama.cpp for the GGUF export adds a few minutes the first time.
!python finetune/colab_train.py --data finetune/dataset/sft_train.jsonl

In [ ]:
#@title 5. Download the Q4_K_M GGUF (save it to finetune/export/qwen-forgesight/ locally)
import glob
from google.colab import files

gguf = glob.glob("finetune/export/**/*.Q4_K_M.gguf", recursive=True) \
    or glob.glob("finetune/export/**/*.gguf", recursive=True)
assert gguf, "No GGUF found — check the training cell output for export errors."
print("Downloading:", gguf[0], f"({os.path.getsize(gguf[0]) / 1e9:.2f} GB)")
files.download(gguf[0])

## Back on your laptop (RTX 3060 + Ollama)

```bash
# 1. Put the downloaded file here (the Modelfile points at this exact path):
#    finetune/export/qwen-forgesight/unsloth.Q4_K_M.gguf
ollama create qwen-forgesight -f finetune/Modelfile

# 2. Measure base vs fine-tune (citation compliance + number fidelity):
python finetune/03_evaluate_vs_base.py --model qwen2.5:3b-instruct
python finetune/03_evaluate_vs_base.py --model qwen-forgesight

# 3. Promote ONLY if the fine-tune wins: set OLLAMA_MODEL=qwen-forgesight and
#    SYNTHESIS_BACKEND=ollama in your local .env. (Railway stays on Groq — no GPU in cloud.)
```

Record the eval JSON in `docs/finetune.md`. The public Railway URL keeps using Groq; the
fine-tuned SLM is the on-prem local path.